In [2]:
import numpy as np
from sklearn.model_selection import KFold

In [3]:
X = np.array([
    [1,2],[3,4],[1,2],[3,4]])

y = np.array([1,2,3,4])

In [5]:
X

array([[1, 2],
       [3, 4],
       [1, 2],
       [3, 4]])

In [7]:
y

array([1, 2, 3, 4])

In [10]:
kf = KFold(n_splits=2)

print(kf.get_n_splits(X))
print(kf)

2
KFold(n_splits=2, random_state=None, shuffle=False)


In [17]:
for train_idx, test_idx in kf.split(X):
    print('train_idx: ',train_idx)
    print('test_idx: ', test_idx)

train_idx:  [2 3]
test_idx:  [0 1]
train_idx:  [0 1]
test_idx:  [2 3]


In [19]:
import pandas as pd

red_url = "https://raw.githubusercontent.com/PinkWink/ML_tutorial/master/dataset/winequality-red.csv"
white_url = "https://raw.githubusercontent.com/PinkWink/ML_tutorial/master/dataset/winequality-white.csv"

red_wine = pd.read_csv(red_url, sep=";")
white_wine = pd.read_csv(white_url, sep=";")

red_wine['color'] = 1.0
white_wine['color'] = 0.

wine = pd.concat([red_wine, white_wine])

x = wine.drop(["color"], axis=1)
y = wine['color']

In [21]:
wine['taste'] = [1. if grade>5 else 0. for grade in wine['quality']]

x = wine.drop(['taste','quality'], axis=1)
y = wine['taste']

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=13)

wine_tree = DecisionTreeClassifier(max_depth=2, random_state=13)
wine_tree.fit(x_train,y_train)

y_pred_tr = wine_tree.predict(x_train)
y_pred_test = wine_tree.predict(x_test)

print('Train ACC: ', accuracy_score(y_train, y_pred_tr))
print('Test ACC: ', accuracy_score(y_test, y_pred_test))

Train ACC:  0.7294593034442948
Test ACC:  0.7161538461538461


In [23]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5)
wine_tree_cv = DecisionTreeClassifier(max_depth=2, random_state=13)

In [25]:
for train_idx, test_idx in kfold.split(x):
    print(len(train_idx), len(test_idx))

5197 1300
5197 1300
5198 1299
5198 1299
5198 1299


In [28]:
cv_accuracy = []

for train_idx, test_idx in kfold.split(x):
    x_train = x.iloc[train_idx]
    x_test = x.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    wine_tree_cv.fit(x_train,y_train)
    pred = wine_tree_cv.predict(x_test)
    cv_accuracy.append(accuracy_score(y_test,pred))

cv_accuracy



[0.6007692307692307,
 0.6884615384615385,
 0.7090069284064665,
 0.7628945342571208,
 0.7867590454195535]

In [30]:
np.mean(cv_accuracy)

0.709578255462782

In [42]:
from sklearn.model_selection import StratifiedKFold

skfold = StratifiedKFold(n_splits=5)
wine_tree_cv = DecisionTreeClassifier(max_depth=2, random_state=13)

cv_accuracy = []

for train_idx, test_idx in StratifiedKFold.split(x,y):
    x_train = x.iloc[train_idx]
    x_test = x.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    wine_tree_cv.fit(x_train,y_train)
    pred = wine_tree_cv.predict(x_test)
    cv_accuracy.append(accuracy_score(y_test,pred))

cv_accuracy

TypeError: split() missing 1 required positional argument: 'y'

In [44]:
from sklearn.model_selection import cross_val_score

skfold = StratifiedKFold(n_splits=5)
wine_tree_cv = DecisionTreeClassifier(max_depth=2, random_state=13)

cross_val_score(wine_tree_cv, x,y, scoring=None, cv=skfold)


array([0.55230769, 0.68846154, 0.71439569, 0.73210162, 0.75673595])

In [45]:
import pandas as pd

red_url = "https://raw.githubusercontent.com/PinkWink/ML_tutorial/master/dataset/winequality-red.csv"
white_url = "https://raw.githubusercontent.com/PinkWink/ML_tutorial/master/dataset/winequality-white.csv"

red_wine = pd.read_csv(red_url, sep=";")
white_wine = pd.read_csv(white_url, sep=";")

red_wine['color'] = 1.0
white_wine['color'] = 0.

wine = pd.concat([red_wine, white_wine])

x = wine.drop(["color"], axis=1)
y = wine['color']

wine['taste'] = [1. if grade>5 else 0. for grade in wine['quality']]

x = wine.drop(['taste','quality'], axis=1)
y = wine['taste']

In [46]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

params = {'max_depth': [2,4,7,10]}

wine_tree = DecisionTreeClassifier(max_depth=2, random_state=13)

In [48]:
gridsearch = GridSearchCV(estimator=wine_tree, param_grid=params, cv=5)
gridsearch.fit(x,y)

GridSearchCV(cv=5,
             estimator=DecisionTreeClassifier(max_depth=2, random_state=13),
             param_grid={'max_depth': [2, 4, 7, 10]})

In [50]:
import pprint

pp = pprint.PrettyPrinter(indent=4)
pp.pprint(gridsearch.cv_results_)

{   'mean_fit_time': array([0.0094018 , 0.01860385, 0.02900476, 0.04320984]),
    'mean_score_time': array([0.00240078, 0.00260105, 0.0022006 , 0.0026011 ]),
    'mean_test_score': array([0.6888005 , 0.66356523, 0.65340854, 0.64401587]),
    'param_max_depth': masked_array(data=[2, 4, 7, 10],
             mask=[False, False, False, False],
       fill_value='?',
            dtype=object),
    'params': [   {'max_depth': 2},
                  {'max_depth': 4},
                  {'max_depth': 7},
                  {'max_depth': 10}],
    'rank_test_score': array([1, 2, 3, 4]),
    'split0_test_score': array([0.55230769, 0.51230769, 0.50846154, 0.51615385]),
    'split1_test_score': array([0.68846154, 0.63153846, 0.60307692, 0.60076923]),
    'split2_test_score': array([0.71439569, 0.72363356, 0.68360277, 0.66743649]),
    'split3_test_score': array([0.73210162, 0.73210162, 0.73672055, 0.71054657]),
    'split4_test_score': array([0.75673595, 0.7182448 , 0.73518091, 0.72517321]),
    'std

In [52]:
gridsearch.best_estimator_

DecisionTreeClassifier(max_depth=2, random_state=13)

In [54]:
gridsearch.best_score_

0.6888004974240539

In [56]:
gridsearch.best_params_

{'max_depth': 2}

In [57]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

estimators = [
    ('scaler', StandardScaler()),
    ('clf', DecisionTreeClassifier())
]

pipe = Pipeline(estimators)

In [60]:
param_grid = [{'clf__max_depth': [2,4,7,10]}]

GridSearch = GridSearchCV(estimator=pipe,param_grid=param_grid, cv=5)
GridSearch.fit(x,y)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('clf', DecisionTreeClassifier())]),
             param_grid=[{'clf__max_depth': [2, 4, 7, 10]}])

In [62]:
GridSearch.best_score_

0.6888004974240539

In [64]:
import pandas as pd

score_df = pd.DataFrame(GridSearch.cv_results_)

In [66]:
score_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf__max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.011602,0.000490,0.002801,0.000400,2,{'clf__max_depth': 2},0.552308,0.688462,0.714396,0.732102,0.756736,0.688800,0.071799,1
1,0.017804,0.000749,0.003201,0.000400,4,{'clf__max_depth': 4},0.512308,0.631538,0.723634,0.732102,0.718245,0.663565,0.083905,2
2,0.031212,0.001169,0.002601,0.000490,7,{'clf__max_depth': 7},0.512308,0.603846,0.678984,0.735181,0.734411,0.652946,0.085167,3
3,0.045612,0.001355,0.002389,0.000477,10,{'clf__max_depth': 10},0.511538,0.605385,0.672055,0.706697,0.722094,0.643554,0.077270,4
